# Task 2: Recommender System

## Introduction

This notebook builds a recommender system using a dataset extracted from Steam, an online
video game distribution service. The dataset contains details of games different members have purchased and played, along
with the number of hours they have played each game and is made available in Unity Catalog.

The objective is to train a collaborative filtering recommender system using MLlib, evaluate its performance, and experiment with multiple configurations to improve the quality of recommendation.


## Data Preprocessing

The raw file was inspected to check if a header row is present before loading the dataset. The output showed that the first row contained actual data, not column names. Default column names ('_c0, _c1, _c2, _c3)  were assigned by spark. Therefore, column names are manually assigned as 'user_id', 'name_of_game, 'behaviour', 'value' based on the description of the dataset.

In [0]:
%python
# inspection of the raw file to see if a header row exists before loading properly
spark.read.option("header", False).csv("/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv").show(3, truncate=False)

+---------+--------------------------+--------+---+
|_c0      |_c1                       |_c2     |_c3|
+---------+--------------------------+--------+---+
|151603712|The Elder Scrolls V Skyrim|purchase|1  |
|151603712|The Elder Scrolls V Skyrim|play    |273|
|151603712|Fallout 4                 |purchase|1  |
+---------+--------------------------+--------+---+
only showing top 3 rows


In [0]:
%python
# Loading the dataset

df = spark.read.csv("/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv", header=False, inferSchema=True)

# assign columns
df = df.withColumnRenamed("_c0", "user_id").withColumnRenamed("_c1", "name_of_game").withColumnRenamed("_c2", "behaviour").withColumnRenamed("_c3", "value")            

# display the schema of the dataframe
df.printSchema()

# display the first 10 rows of the dataframe
df.show(10)
       



root
 |-- user_id: integer (nullable = true)
 |-- name_of_game: string (nullable = true)
 |-- behaviour: string (nullable = true)
 |-- value: double (nullable = true)

+---------+--------------------+---------+-----+
|  user_id|        name_of_game|behaviour|value|
+---------+--------------------+---------+-----+
|151603712|The Elder Scrolls...| purchase|  1.0|
|151603712|The Elder Scrolls...|     play|273.0|
|151603712|           Fallout 4| purchase|  1.0|
|151603712|           Fallout 4|     play| 87.0|
|151603712|               Spore| purchase|  1.0|
|151603712|               Spore|     play| 14.9|
|151603712|   Fallout New Vegas| purchase|  1.0|
|151603712|   Fallout New Vegas|     play| 12.1|
|151603712|       Left 4 Dead 2| purchase|  1.0|
|151603712|       Left 4 Dead 2|     play|  8.9|
+---------+--------------------+---------+-----+
only showing top 10 rows


The datset contains four columns: **user_id**, stored as integer, **name_of_game**, stored as string, **behaviour**, stored as string and **value**, stored as double.

I checked for null values in each column using the code in cell 7. The code returns the count of rows with null values in each column. This check is important, especially for the 'user_id' and 'name_of_game' columns because ALS requires valid ID for for each record. 

In [0]:
%python
# check for null values in the columns of the dataframe
from pyspark.sql.functions import count, when, isnull
df.select([count(when(isnull(c), c)).alias(c) for c in df.columns]).show()
       


+-------+------------+---------+-----+
|user_id|name_of_game|behaviour|value|
+-------+------------+---------+-----+
|      0|           0|        0|    0|
+-------+------------+---------+-----+



The output confirms that there are no null values across the columns in the dataset. Therefore, I can use all records in the analysis.

### Preprocessing Decisions

I used only 'play' interactions for the recommender system because playtime reflects user preference better than purchase. Purchase does not necessarily show engagement. To support this decision, I compared the number of users with play records with those with purchase records. **12,393** users have purchase records and **11,350** have play records. I queried further to check the number of users who have only purchase records with no play data recorded. There are **1,043** such users, which is approximately **8%** of the users. 

This means that most users who purchase games also play them. Therefore, taking out users with only purchase interactions leads to only a small loss of user data.

In [0]:
%python
# Check how many users have play records vs purchase only

from pyspark.sql import functions as F
df_play_users = df.filter(F.col("behaviour") == "play").select("user_id").distinct()
df_purchase_users = df.filter(F.col("behaviour") == "purchase").select("user_id").distinct()

print(f"Users with play data: {df_play_users.count():,}")
print(f"Users with purchase data: {df_purchase_users.count():,}")

Users with play data: 11,350
Users with purchase data: 12,393


In [0]:
%python
#find users who have purchase records but no play records at all
purchase_only_users = df_purchase_users.subtract(df_play_users)
print(f"Users with purchase only (no play data): {purchase_only_users.count():,}")

Users with purchase only (no play data): 1,043


In [0]:
%python
# filter the data to only include only play interactions
df_play_interactions = df.filter(df.behaviour == "play").select("user_id", "name_of_game", "value").distinct()
print(f"Number of play interactions: {df_play_interactions.count():,}")
       


Number of play interactions: 70,489


To run Alternating Least Squares (ALS) matrix factorization using MLlib,  both users and items must have integerID values. Therefore, I converted string values into indexed numerical representations.

In [0]:
%python
from pyspark.ml.feature import StringIndexer

# Index the 'name_of_game' column
game_indexer = StringIndexer(inputCol="name_of_game", outputCol="name_of_game_indexed")

# Index the 'user_id' column
user_indexer = StringIndexer(inputCol="user_id", outputCol="user_id_indexed")

indexed_df = game_indexer.fit(df_play_interactions).transform(df_play_interactions)
indexed_df = user_indexer.fit(indexed_df).transform(indexed_df)
       
df_final = indexed_df.select("user_id_indexed", "name_of_game_indexed",  "value") 

df_final.show()

+---------------+--------------------+-----+
|user_id_indexed|name_of_game_indexed|value|
+---------------+--------------------+-----+
|         1254.0|               249.0|  1.9|
|         6171.0|               381.0|  0.6|
|         2637.0|                24.0| 86.0|
|          432.0|                50.0|147.0|
|         1102.0|               949.0| 21.0|
|         1102.0|               162.0|  0.7|
|         6631.0|               126.0|  0.4|
|           69.0|              1557.0|  3.5|
|           67.0|                13.0|  1.3|
|          523.0|               524.0| 17.2|
|          523.0|                83.0|  1.1|
|           91.0|               444.0|  2.1|
|          674.0|                 1.0| 13.5|
|          109.0|                10.0|  8.6|
|           80.0|              1076.0|  3.1|
|           80.0|              2573.0|  1.7|
|         5991.0|                 0.0|  4.1|
|         1243.0|                 1.0| 22.0|
|          125.0|                 2.0|  4.7|
|        1


## Exploratory Data Analysis

I carried out some initial exploratory data analysis of the filtered play dataset to better understand how game polularity and behaviour are distributed before training the recommender system.

I explored the following aspects:
- The dataset statistics: number of users, number of games, and number of interactions.
- Most popular games by number of players
- Top games by total hours played
- Distribution of playtime across all records
- Distribution of games played per user


In [0]:
%python
# the statistics of the dataset

print(f"Total number of play interactions: {df_play_interactions.count():,}")
print(f"Number of unique users: {df_play_interactions.select("user_id").distinct().count():,}")
print(f"Number of unique games: {df_play_interactions.select("name_of_game").distinct().count():,}")




Total number of play interactions: 70,489
Number of unique users: 11,350
Number of unique games: 3,600


In [0]:
%python
#calculate matrix sparsity percentage
total_possible = df_play_interactions.select("user_id").distinct().count() * df_play_interactions.select("name_of_game").distinct().count()
total_interactions = df_play_interactions.count()
sparsity = (1 - total_interactions / total_possible) * 100
print(f"Sparsity percentage: {sparsity:.2f}%")
       


Sparsity percentage: 99.83%


The user-item matrix has a sparsity of **99.83%**, which means the dataset is highy sparse. This high sparsity indicates that most users interact with minority of the games.

In [0]:
%python
# Most popular games by number of players who have played them
from pyspark.sql.functions import count, countDistinct

df_popular_games = df_play_interactions.groupBy("name_of_game").agg(countDistinct("user_id").alias("player_count")).orderBy("player_count", ascending=False)

df_popular_games.show(10)

+--------------------+------------+
|        name_of_game|player_count|
+--------------------+------------+
|              Dota 2|        4841|
|     Team Fortress 2|        2323|
|Counter-Strike Gl...|        1377|
|            Unturned|        1069|
|       Left 4 Dead 2|         801|
|Counter-Strike So...|         715|
|The Elder Scrolls...|         677|
|         Garry's Mod|         666|
|      Counter-Strike|         568|
|Sid Meier's Civil...|         554|
+--------------------+------------+
only showing top 10 rows


The output shows that the most popular game based on the number of distinct players is **Dota 2** with **4,841 players**. Tt has more than twice the number of players of **Team Fortress 2**, which is the second most popular game and an even wider gap from the other games on the list. Looking at the games and the numbers, popularity appears to be concentrated among a few established games.

In [0]:
%python

import pyspark.sql.functions as F

# Top games by total hours played across all users
# Used round function to round the total hours to 2 decimal places

df_top_games = df_play_interactions.groupBy("name_of_game").agg(F.round(F.sum("value"), 2).alias("total_hours_played")).orderBy("total_hours_played", ascending=False)

df_top_games.show(10)
       


+--------------------+------------------+
|        name_of_game|total_hours_played|
+--------------------+------------------+
|              Dota 2|          981684.6|
|Counter-Strike Gl...|          322771.6|
|     Team Fortress 2|          173673.3|
|      Counter-Strike|          134261.1|
|Sid Meier's Civil...|           99821.3|
|Counter-Strike So...|           96075.5|
|The Elder Scrolls...|           70889.3|
|         Garry's Mod|           49725.3|
|Call of Duty Mode...|           42009.9|
|       Left 4 Dead 2|           33596.7|
+--------------------+------------------+
only showing top 10 rows


The output shows that **Dota 2** has the highest playtime with **981,684** hours, suggesting that not only does the game have the most number of players, the players spend more time on it compared to other games in the dataset.

Although **Team Fortress 2 ** is the second most popular game, followed by **Counter-Strike Global Offensive**, the output shows that **Counter-Strike Global Offensive** has more playtime than **Team Fortress 2**. This suggests that **Counter-Strike** players spend more hours per person on the game than **Team Fortress 2 players**.

In [0]:
%python
# distribution by hours played
df_play_interactions.select("value").summary("min", "25%", "50%", "75%", "max", "mean").show()
       
       


+-------+----------------+
|summary|           value|
+-------+----------------+
|    min|             0.1|
|    25%|             1.0|
|    50%|             4.5|
|    75%|            19.1|
|    max|         11754.0|
|   mean|48.8780632439103|
+-------+----------------+




The median playtime is 4.5 hours and the mean is 11,754 hours, which indicates skewness to the right. The mean suggests that a small number of users have very high playtimes and are increasing the mean significantly.

The interquartile range(1.0-19.1 hours) shows that most users have relatively moderate playtimes.


In [0]:
%python

from pyspark.sql.functions import count
# distribution of games per user

df_games_per_user = df_play_interactions.groupBy("user_id").agg(count("name_of_game").alias("game_count")).select("game_count").summary("min", "25%", "50%", "75%", "max", "mean").show()

+-------+-----------------+
|summary|       game_count|
+-------+-----------------+
|    min|                1|
|    25%|                1|
|    50%|                1|
|    75%|                3|
|    max|              498|
|   mean|6.210484581497798|
+-------+-----------------+



The distribution of games played per user is also skewed to the right. The median user has played just one game. 75% of users have played 3 or fewer games.The maximum is 498 games and the mean is approximately 6.2 games per user. This result suggests a low history of interaction by the majority of users.


##Model Training
I applied a log transformation using log1p() on the `value` column to compress the range and reduce the influence of extreme outliers identified in hours played. After this, the data is split into training and test sets in the ratio 80:20 respectively using a random split with a fixed seed for reproducibility.

In [0]:
%python

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
import mlflow
from pyspark.sql import functions as F


#cast ID columns to integer and apply log transformation to hours played

df_model =df_final.select(
    F.col("user_id_indexed").cast("int").alias("userid"),
    F.col("name_of_game_indexed").cast("int").alias("nameofgame"),
    
    #log transformation
    F.log1p(F.col("value")).alias("Implicit_score")

)

print(f"Model dataset rows: {df_model.count()}")
df_model.show(5)


Model dataset rows: 70489
+------+----------+------------------+
|userid|nameofgame|    Implicit_score|
+------+----------+------------------+
|  1254|       249|1.0647107369924282|
|  6171|       381|0.4700036292457356|
|  2637|        24| 4.465908118654584|
|   432|        50| 4.997212273764115|
|  1102|       949| 3.091042453358316|
+------+----------+------------------+
only showing top 5 rows


In [0]:
%python
#train/test split
#split the data into 80% training and 20%test sets

train, test = df_model.randomSplit([0.8, 0.2], seed=42)
       
print(f"Training dataset rows: {train.count()}")
print(f"Test dataset rows: {test.count()}")
       


Training dataset rows: 56527
Test dataset rows: 13962


In [0]:
%python
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow

#set MLflow experiment
experiment_name = ("/Shared/ALS_Steam_Recommender")
mlflow.set_experiment(experiment_name)


#Train initial ALS model with default hyperparameters
with mlflow.start_run(run_name="ALS") as run:
  
  als = ALS(
    userCol="userid",
    itemCol="nameofgame",
    ratingCol="Implicit_score",
    rank=10,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    seed=42
  )
  
  als_model = als.fit(train)
  
  

  #generate predictions on test set
  predictions = als_model.transform(test)

  #evaluate model on test set using RMSE
  evaluator = RegressionEvaluator(metricName="rmse", labelCol="Implicit_score", predictionCol="prediction")
  rmse = evaluator.evaluate(predictions)
  
  
#log params and metrics to MLflow
mlflow.log_param("rank", 10)
mlflow.log_param("maxIter", 10)
mlflow.log_param("regParam", 0.1)
mlflow.log_metric("rmse", rmse)

print(f"initial model RMSE: {rmse:.4f}")

initial model RMSE: 1.5054


## Hyperparameter Tuning

A RMSE of **1.5054** was achieved by the initial model, using default hyperparameters (`rank=10`, `maxIter=10`, `regParam=0.1`). A grid search was conducted over `rank` (the number of latent factors in the matrix factorisation, tested at values of 10, 20 and 30. A higher rank can capture more complex patterns, but increases the risk of overfitting. ) `regParam` (the regularisation parameter that penalises model complexity to prevent overfitting, tested at values of 0.01, 0.1, and 1.0) and `maxIter`(fixed at 10 for all runs for consistency) to improve performance.

9 combinations are evaluated, with each run tracked with **MLflow**, hyperparameters and RMSE logged for every combination to allow easy comparison of results.

In [0]:

%python
import mlflow
mlflow.end_run()

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow

#set MLflow experiment
experiment_name = ("/Shared/ALS_Steam_Recommender")
mlflow.set_experiment(experiment_name)

# define hyperparameter grid to search
ranks = [10, 20, 30]
regParams = [0.01, 0.1, 1.0]

evaluator = RegressionEvaluator(metricName="rmse", labelCol="implicit_score", predictionCol="prediction")

best_rmse = float("inf")
best_params = {}

# iterate over all hyperparameter combinations and train models
for rank in ranks:
    for regParam in regParams:
        with mlflow.start_run(run_name=f"ALS_{rank}_{regParam}") as run:
           
            als = ALS(
                userCol="userid",
                itemCol="nameofgame",
                ratingCol="Implicit_score",
                rank=rank,
                maxIter=10,
                regParam=regParam,
                coldStartStrategy="drop",
                seed=42
            )
            als_model = als.fit(train)
            predictions = als_model.transform(test)
            rmse = evaluator.evaluate(predictions)

            #log params and metrics to MLflow
            mlflow.log_param("rank", rank)
            mlflow.log_param("maxIter", 10)
            mlflow.log_param("regParam", regParam)
            mlflow.log_metric("rmse", rmse)

            print(f"rank: {rank}, regParam: {regParam}, rmse: {rmse:.4f}")
            
            #keep track of best model
            if rmse < best_rmse:
                best_rmse = rmse
                best_params = {"rank": rank, "regParam": regParam}
                
print(f"\nBest RMSE: {best_rmse:.4f}")
print(f"Best params: {best_params}")






rank: 10, regParam: 0.01, rmse: 1.9804
rank: 10, regParam: 0.1, rmse: 1.5054
rank: 10, regParam: 1.0, rmse: 1.6577
rank: 20, regParam: 0.01, rmse: 1.8785
rank: 20, regParam: 0.1, rmse: 1.4820
rank: 20, regParam: 1.0, rmse: 1.6576
rank: 30, regParam: 0.01, rmse: 1.7732
rank: 30, regParam: 0.1, rmse: 1.4620
rank: 30, regParam: 1.0, rmse: 1.6577

Best RMSE: 1.4620
Best params: {'rank': 30, 'regParam': 0.1}


The grid search results show that the best model achieved an RMSE of **1.4620** with `rank=30` and `regParam=0.1`, which is an improvement over the initial model's RMSE of 1.5054.

From the patterns visible in the results, **`regParam`**=0.01 has the worst performance across all rank values. This suggests that overfitting occurs when regularisation is too low. **`regParam`=0.1** has the best performance across all rank values. It is the optimal regularisation strength for this dataset.

The best model (`rank=30`, `regParam=0.1`) will be retrained and used for recommendations generation

## Model Evaluation

The best hyperparameter combination from the grid search was `rank=30` and `regParam=0.1`, with an RMSE 0f **1.4620 **.
This model is retrained on the training set and evaluated on the test set before being used for recommendations generation.

In [0]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow

#set MLflow experiment
experiment_name = ("/Shared/ALS_Steam_Recommender")
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="ALS_best_model") as run:
  
  # retrain model with best hyperparameters from grid search
  
  best_als = ALS(
    userCol="userid",
    itemCol="nameofgame",
    ratingCol="Implicit_score",
    rank=30,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop",
    seed=42
  )
  
  best_als_model = best_als.fit(train)
  
  

  #generate predictions on test set
  predictions = best_als_model.transform(test)

  #evaluate model on test set using RMSE
  evaluator = RegressionEvaluator(metricName="rmse", labelCol="Implicit_score", predictionCol="prediction")
  rmse = evaluator.evaluate(predictions)
  
  
#log params and metrics to MLflow
mlflow.log_param("rank", 30)
mlflow.log_param("maxIter", 10)
mlflow.log_param("regParam", 0.1)
mlflow.log_metric("rmse", rmse)

print(f"best model RMSE: {rmse:.4f}")


best model RMSE: 1.4620


## Generating Recommendations

The best model is used to generate the top 10 game recommendations for a sample of users. The indexed game names that were earlier created due to ALS only allowing integer IDs is mapped back to the actual game names using the `StringIndexer` labels. This will allow the recommendations to be meaningfully interpreted.

In [0]:
# Generate recommendations for all users
recommendations = best_als_model.recommendForAllUsers(10)

# The StringIndexer model was used to convert the game names to integer IDs. We need to reverse this transformation to get the original game names.
game_labels= game_indexer.fit(df_play_interactions).labels

# Create a DataFrame mapping the integer IDs back to game names.
game_id_to_name = spark.createDataFrame([(i, game_labels[i]) for i in range(len(game_labels))],["nameofgame", "name_of_game"]
)

#Explode the recommendations DataFrame to get one row per user and game.
from pyspark.sql.functions import explode, col

recs_exploded = recommendations.select(col("userId"), explode(col("recommendations")).alias("recommendation")). select ("userid",  col("recommendation.nameofgame").alias("nameofgame"), col("recommendation.rating"). alias("predicted_score")
)
#Join the recommendations DataFrame with the game_id_to_name DataFrame to get the original game names.
recommendations_with_names = recs_exploded.join(game_id_to_name, on="nameofgame", how="left")\
    .select("userid", "name_of_game", "predicted_score") \
    .orderBy("userid", "predicted_score", ascending=[True, False])

recommendations_with_names.show(50)



+------+--------------------+---------------+
|userid|        name_of_game|predicted_score|
+------+--------------------+---------------+
|     0|Age of Empires On...|       4.389606|
|     0|Total War ROME II...|      3.5457046|
|     0|Might & Magic Her...|      3.5044577|
|     0|        Soul Gambler|       3.498454|
|     0|Assassin's Creed ...|       3.430248|
|     0|Pro Evolution Soc...|      3.3802881|
|     0|The Witcher 3 Wil...|      3.2542734|
|     0|Phantom Breaker B...|      3.2378888|
|     0|       Ancient Space|      3.2219532|
|     0|Counter-Strike Gl...|      3.2048895|
|     1|          EVE Online|      3.9751813|
|     1|            NBA 2K14|      3.7880397|
|     1|     Elite Dangerous|      3.6193485|
|     1|METAL GEAR SOLID ...|      3.5133934|
|     1|           Fallout 4|       3.477207|
|     1|The Elder Scrolls...|      3.4720984|
|     1|Age of Empires On...|      3.4203928|
|     1|Football Manager ...|      3.3651412|
|     1|FINAL FANTASY XIV...|     

The model generates top-10 recommendations for each user based on predicted preference scores. The predicted scores are derived from patterns in user-item interaction and it represents the likelihood that a user will engage with a game. The higher the score, the higher the likelihood that the user will engage with the game.  The output shows that users are recommended games with similarity to games preferred by users with similar behavior, which demonstrates the effectiveness of collaborative filtering. For example, User 0 is recommended a set of games, including Age of Empires, Total War ROME II, Assassin's creed, which suggests that the model identified a prefernce for this genre. User 1 has a different set of games recommended, which indicates that the model generates personalised recommendations.


## Conclusion
A collaborative filtering recommender system was built using the steam dataset, which contains implicit feedback data on game play behavior for 11,350 users across 3600 unique games.

In the data preprocessing, a decison was made to only use 'play' behavior records for training as this is a better indicator of user engagement. 1,043 users (appproximately 8% of the dataset), who had only purchase record were excluded.

Play hours were transformed into log using log1p() before modelling to handle the right skew identified during exploratory data analysis where the mean play hours(48.88 hours) was significantly greater than the median(4.5 hours).

A RMSE of 1.5054 was achieved by the initial ALS model. A grid search over `rank` (10, 20, 30) and 'regParam'(0.01, 0.1, 1.0) identified `rank=30` and `regParam=0.1` as the best hyperparameter combination, with an improved RMSE of **1.4620**. All runs were tracked with **MLflow**.

The model generates personalised recommendations to users based on the patterns of user-item interacton rather than just recommending the games that are popular to all users. 